# End-to-End RAG 평가 구축

이 노트북은 완전한 **RAG(Retrieval-Augmented Generation, 검색 증강 생성)** 파이프라인을 구축하고 평가하는 **단계별 실습**입니다.

다음을 학습합니다:

1. PDF에서 end-to-end RAG 파이프라인 구축
2. 평가용 **ground truth(정답 기준)** 데이터셋 생성
3. **Faithfulness(충실도)**와 **Answer Relevance(답변 관련성)**를 측정해 환각(hallucination) 감소
4. 결과 해석 및 파이프라인 반복 개선

> 권장 워크플로: 셀을 하나씩 실행하고, 설명을 읽은 뒤 출력을 확인한 다음 다음 단계로 진행하세요.


## Step 0 — 의존성 설치

이 워크숍에서 사용하는 도구:

- **LangChain** — 파이프라인 구축
- **Chroma** — 벡터 데이터베이스
- **Ragas** — RAG 평가 지표
- **Ollama** — RAG 답변 생성: 노트북이 `subprocess`로 `ollama_model_runner.py`를 호출합니다 (`Module_4_Building_a_comprehensive_RAG_system.ipynb`와 동일 패턴)
- **Sentence Transformers** — `langchain-huggingface` / `HuggingFaceEmbeddings`로 인덱싱 및 Ragas에 사용 (기본값 `sentence-transformers/all-MiniLM-L6-v2`, Module 4 실습과 동일)

이미 설치했다면 다음 셀을 건너뛰어도 됩니다.


In [1]:
# Run once per environment
%pip install -q -U langchain langchain-community langchain-openai langchain-huggingface sentence-transformers chromadb pypdf pandas numpy tqdm ragas datasets

# ragas 0.4.3 imports Vertex AI from a removed langchain-community path.
# Patch the installed package so `import ragas` works with modern LangChain.
import importlib.util
from pathlib import Path

if importlib.util.find_spec("ragas") is not None:
    import ragas

    base_py = Path(ragas.__file__).parent / "llms" / "base.py"
    text = base_py.read_text()
    marker = "_VERTEX_COMPLETION_TYPES"
    if marker not in text:
        text = text.replace(
            "from langchain_community.chat_models.vertexai import ChatVertexAI\n"
            "from langchain_community.llms import VertexAI\n",
            "",
        )
        text = text.replace(
            "MULTIPLE_COMPLETION_SUPPORTED = [\n"
            "    OpenAI,\n"
            "    ChatOpenAI,\n"
            "    AzureOpenAI,\n"
            "    AzureChatOpenAI,\n"
            "    ChatVertexAI,\n"
            "    VertexAI,\n"
            "]",
            """_VERTEX_COMPLETION_TYPES: list = []
try:
    from langchain_google_vertexai import ChatVertexAI
    from langchain_google_vertexai import VertexModel as VertexAI

    _VERTEX_COMPLETION_TYPES = [ChatVertexAI, VertexAI]
except ImportError:
    try:
        from langchain_community.chat_models.vertexai import ChatVertexAI
        from langchain_community.llms import VertexAI

        _VERTEX_COMPLETION_TYPES = [ChatVertexAI, VertexAI]
    except ImportError:
        pass

MULTIPLE_COMPLETION_SUPPORTED = [
    OpenAI,
    ChatOpenAI,
    AzureOpenAI,
    AzureChatOpenAI,
    *_VERTEX_COMPLETION_TYPES,
]""",
        )
        base_py.write_text(text)
        print("ragas Vertex AI import 호환성 패치를 적용했습니다.")


Note: you may need to restart the kernel to use updated packages.


## Step 1 — Import 및 환경 설정

이 워크숍은 명시적이고 투명하게 진행합니다:

- 동작마다 작은 코드 블록 하나
- 빠른 sanity check를 위한 print 출력
- 재사용 가능한 헬퍼 함수


In [2]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

pd.set_option("display.max_colwidth", 150)


### 모델 엔드포인트 설정

**RAG 생성(답변)** 은 이 폴더의 `ollama_model_runner.py`(또는 작업 디렉터리가 repo 루트이면 `Module_4/ollama_model_runner.py`)를 실행하는 **Ollama**를 사용합니다. 노트북 밖에서 `ollama serve`와 `ollama pull <model>`을 실행하세요.

**예시** python Module_4/ollama_model_runner.py --host http://localhost:11434 --models llama3.1:8b --prompt-file /tmp/tmpivn3qis0.txt --temperature 0.2 --max-tokens 320

**Embeddings(임베딩)** (Chroma 인덱스 및 Ragas)는 LangChain의 `HuggingFaceEmbeddings`를 통해 **Sentence Transformers**를 사용하며, `Module_4_Building_a_comprehensive_RAG_system.ipynb`와 맞춰져 있습니다 (기본 모델명: `sentence-transformers/all-MiniLM-L6-v2`). 다른 HF 모델 id를 쓰려면 환경 변수 `RAG_EMBED_MODEL`로 덮어쓰세요.

> Ollama 호스트, chat 모델, embedding 모델 id가 다르면 다음 두 코드 셀을 수정하세요.


In [3]:
# --- Ollama: used by ollama_model_runner.py for RAG answers (subprocess) ---
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODELS = os.getenv("OLLAMA_MODELS", "llama3.1:8b")  # comma-separated if you ever pass several
OLLAMA_TEMPERATURE = float(os.getenv("OLLAMA_TEMPERATURE", "0.2"))
OLLAMA_MAX_TOKENS = int(os.getenv("OLLAMA_MAX_TOKENS", "512"))

# --- Sentence Transformers: same default as Module_4_Building_a_comprehensive_RAG_system.ipynb ---
EMBED_MODEL_NAME = os.getenv("RAG_EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")


def _resolve_ollama_runner_path() -> Path:
    for candidate in (Path("ollama_model_runner.py"), Path("Module_4") / "ollama_model_runner.py"):
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "ollama_model_runner.py not found next to the notebook or under Module_4/. "
        "Open this notebook from Module_4 or clone the file into your cwd."
    )


OLLAMA_RUNNER = _resolve_ollama_runner_path()
OLLAMA_CHAT_MODEL = OLLAMA_MODELS.split(",")[0].strip()

print("OLLAMA_HOST:", OLLAMA_HOST)
print("OLLAMA_RUNNER:", OLLAMA_RUNNER)
print("OLLAMA_MODELS:", OLLAMA_MODELS)
print("OLLAMA_CHAT_MODEL (Ragas -> Ollama /v1):", OLLAMA_CHAT_MODEL)
print("EMBED_MODEL_NAME (Sentence Transformers):", EMBED_MODEL_NAME)


OLLAMA_HOST: http://localhost:11434
OLLAMA_RUNNER: /home/ethan/newgen/KMOU_Course/Module_4/ollama_model_runner.py
OLLAMA_MODELS: llama3.1:8b
OLLAMA_CHAT_MODEL (Ragas -> Ollama /v1): llama3.1:8b
EMBED_MODEL_NAME (Sentence Transformers): sentence-transformers/all-MiniLM-L6-v2


In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_NAME,
    encode_kwargs={"normalize_embeddings": True},
)

# Ragas calls an LLM over HTTP; use Ollama's OpenAI-compatible endpoint (same chat model as the runner).
ragas_llm = ChatOpenAI(
    model=OLLAMA_CHAT_MODEL,
    api_key=os.getenv("OPENAI_API_KEY", "ollama"),
    base_url=f"{OLLAMA_HOST.rstrip('/')}/v1",
    temperature=0,
)


def call_ollama_runner(prompt: str, *, models: str | None = None) -> str:
    """Run ollama_model_runner.py (same argv style as Module_4_Building_a_comprehensive_RAG_system.ipynb).

    Pass ``models=`` to override ``OLLAMA_MODELS`` (for example a dedicated labeler model in Step 6b).
    """
    models_arg = models if models is not None else OLLAMA_MODELS
    pf = tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8")
    path = pf.name
    try:
        pf.write(prompt)
        pf.close()
        cmd = [
            sys.executable,
            str(OLLAMA_RUNNER),
            "--host",
            OLLAMA_HOST,
            "--models",
            models_arg,
            "--prompt-file",
            path,
            "--temperature",
            str(OLLAMA_TEMPERATURE),
            "--max-tokens",
            str(OLLAMA_MAX_TOKENS),
        ]
        run = subprocess.run(cmd, capture_output=True, text=True)
        if run.returncode != 0:
            raise RuntimeError(
                f"ollama_model_runner.py exit {run.returncode}\nSTDERR:\n{run.stderr[:4000]}"
            )
        try:
            payload = json.loads(run.stdout)
        except json.JSONDecodeError as e:
            raise RuntimeError(
                f"Invalid JSON from ollama_model_runner.py: {e}\nSTDOUT:\n{run.stdout[:2000]}"
            ) from e
        outputs = payload.get("outputs") or []
        if not outputs:
            raise RuntimeError("Runner JSON contained no outputs")
        first = outputs[0]
        if first.get("status") != "ok":
            raise RuntimeError(f"Ollama runner error: {first.get('status')}")
        return str(first.get("response", "")).strip()
    finally:
        try:
            os.unlink(path)
        except OSError:
            pass


print("Embeddings(HuggingFace), ragas_llm, call_ollama_runner() 준비 완료.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings(HuggingFace), ragas_llm, call_ollama_runner() 준비 완료.


## Step 2 — PDF 문서 로드 및 검사

다음 파일을 로드합니다: `data/IGF Code (124Pages).pdf`.

이 단계에서는 다음만 검증합니다:

- 파일 존재 여부
- 로드된 페이지 수
- 샘플 텍스트 품질


In [5]:
pdf_path = Path("data") / "IGF Code (124Pages).pdf"
print("PDF 경로:", pdf_path)
print("존재 여부:", pdf_path.exists())


PDF 경로: data/IGF Code (124Pages).pdf
존재 여부: True


In [6]:
loader = PyPDFLoader(str(pdf_path))
docs = loader.load()
print("로드된 페이지 수:", len(docs))


로드된 페이지 수: 124


In [7]:
sample_page = 0
print(docs[sample_page].page_content[:1500])


MSC 95/22/Add.1 
Annex 1, page 1 
 
 
I:\MSC\95\MSC 95-22-ADD.1 (E).docx 
ANNEX 1 
 
RESOLUTION MSC.391(95)  
(adopted on 11 June 2015) 
 
ADOPTION OF THE INTERNATIONAL CODE OF SAFETY FOR SHIPS USING GASES 
OR OTHER LOW-FLASHPOINT FUELS (IGF CODE) 
 
 
THE MARITIME SAFETY COMMITTEE, 
 
RECALLING Article 28(b) of the Convention on the International Maritime Organization 
concerning the function of the Committee, 
 
RECOGNIZING the need for a mandatory code for ships using gases or other low -flashpoint 
fuels, 
 
NOTING resolution MSC .392(95), by which it adopted, inter alia,  amendments to 
chapters II-1,II-2 and the appendix to the annex of the International Convention for the Safety 
of Life at Sea, 1974  ("the Convention"), to make the provisions of the International Code of 
Safety for Ships using Gases or other  Low-flashpoint Fuels (IGF Code) mandatory under the 
Convention, 
 
HAVING CONSIDERED, at its ninety-fifth session, the draft International Code of Safety for 
Ships usin

## Step 3 — 검색용 Chunking(청킹) 전략

RAG 품질은 chunking에 크게 좌우됩니다.

실용적인 기본값으로 시작합니다:

- `chunk_size = 1000`
- `chunk_overlap = 150`

이후 이 값을 조정하고 평가 점수를 비교할 수 있습니다.


In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(docs)
print("총 chunk 수:", len(chunks))


총 chunk 수: 400


In [9]:
ROW_NUMBER = 20
chunk_preview = pd.DataFrame(
    {
        "chunk_id": list(range(min(ROW_NUMBER, len(chunks)))),
        "page": [chunks[i].metadata.get("page") for i in range(min(ROW_NUMBER, len(chunks)))],
        "text_preview": [chunks[i].page_content[:220].replace("\n", " ") for i in range(min(ROW_NUMBER, len(chunks)))],
    }
)
chunk_preview


,chunk_id,page,text_preview
0,0,0,"MSC 95/22/Add.1 Annex 1, page 1 I:\MSC\95\MSC 95-22-ADD.1 (E).docx ANNEX 1 RESOLUTION MSC.391(95) (adopted on 11 June 2015) ADOPTIO..."
1,1,0,"Convention, HAVING CONSIDERED, at its ninety-fifth session, the draft International Code of Safety for Ships using Gases or other Low-flashpoi..."
2,2,0,"5 REQUESTS the Secretary-General of the Organization to transmit certified copies of the present resolution and the text of the IGF Code, contain..."
3,3,1,"MSC 95/22/Add.1 Annex 1, page 2 I:\MSC\95\MSC 95-22-ADD.1 (E).docx ANNEX INTERNATIONAL CODE OF SAFETY FOR SHIPS USING GASES OR OTHER LO..."
4,4,1,2.3 Alternative design ................................................................................................ 11 3 GOAL AND FUNCTIONAL ...
5,5,1,4.3 Limitation of explosion consequences .............................................................. 13 PART A-1 SPECIFIC REQUIREMENTS FOR SHI...
6,6,1,5.6 Regulations for ESD-protected machinery spaces ........................................... 18 5.7 Regulations for location and protection of ...
7,7,1,6 FUEL CONTAINMENT SYSTEM ............................................................................ 21 6.1 Goal .................................
8,8,2,"MSC 95/22/Add.1 Annex 1, page 3 I:\MSC\95\MSC 95-22-ADD.1 (E).docx 6.2 Functional requirements ..............................................."
9,9,2,6.9 Regulations for the maintaining of fuel storage condition .............................. 62 6.10 Regulations on atmospheric control within th...


## Step 4 — 벡터 인덱스 구축 또는 로드 (Chroma)

chunk는 `outputs/chroma_igf_rag_eval/` 아래 **로컬** Chroma 데이터베이스에 embedding되어 저장됩니다.

- **첫 실행(폴더 없음/비어 있음):** `chunks`에서 인덱스를 구축한 뒤 디스크에 저장합니다.
- **이후 실행:** 디스크의 기존 인덱스를 로드해 re-embedding을 건너뜁니다 (훨씬 빠름).
- **처음부터 재구축:** 다음 코드 셀에서 `FORCE_REBUILD_CHROMA = True`로 설정하거나 환경 변수 `FORCE_REBUILD_CHROMA=1`을보냅니다 (chunking, PDF, embedding 모델을 바꿔 벡터 일관성을 맞출 때 필요).

**로드** 경로에서는 메모리에 `chunks`가 없어도 됩니다. 저장 시와 동일한 `embeddings` 객체(동일 모델)가 필요합니다. **구축** 경로에서는 `chunks`와 `embeddings`를 정의한 이전 셀을 먼저 실행하세요.


In [10]:
import shutil

CHROMA_COLLECTION = "igf_code_rag_eval"
chroma_dir = Path("outputs") / "chroma_igf_rag_eval"

# Set True in-notebook to wipe disk and rebuild from `chunks` (or export FORCE_REBUILD_CHROMA=1).
FORCE_REBUILD_CHROMA = False
FORCE_REBUILD_CHROMA = FORCE_REBUILD_CHROMA or os.getenv("FORCE_REBUILD_CHROMA", "0").lower() in (
    "1",
    "true",
    "yes",
)


def _chroma_persist_ready(path: Path) -> bool:
    """True if a Chroma persistent store already exists on disk."""
    if not path.is_dir():
        return False
    return (path / "chroma.sqlite3").is_file()


chroma_dir.mkdir(parents=True, exist_ok=True)

if FORCE_REBUILD_CHROMA and chroma_dir.exists():
    shutil.rmtree(chroma_dir)
    chroma_dir.mkdir(parents=True, exist_ok=True)

if _chroma_persist_ready(chroma_dir) and not FORCE_REBUILD_CHROMA:
    vector_db = Chroma(
        collection_name=CHROMA_COLLECTION,
        embedding_function=embeddings,
        persist_directory=str(chroma_dir),
    )
    n = vector_db._collection.count()
    print(f"디스크에서 기존 Chroma DB를 로드했습니다 ({n}개 문서).")
else:
    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=CHROMA_COLLECTION,
        persist_directory=str(chroma_dir),
    )
    print(f"chunk에서 Chroma DB를 구축해 디스크에 저장했습니다 ({len(chunks)}개 문서).")

print("Chroma 디렉터리:", chroma_dir.resolve())


/tmp/ipykernel_336672/3022736020.py:29: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


디스크에서 기존 Chroma DB를 로드했습니다 (400개 문서).
Chroma 디렉터리: /home/ethan/newgen/KMOU_Course/Module_4/outputs/chroma_igf_rag_eval


In [11]:
retriever = vector_db.as_retriever(search_kwargs={"k": 4})
print("Retriever top-k:", retriever.search_kwargs.get("k"))


Retriever top-k: 4


## Step 5 — 간단한 RAG 답변 함수 구축

최소하지만 견고한 RAG 함수를 정의합니다:

1. top-k chunk 검색
2. context 블록 구성
3. 모델에게 **context에서만** 답변하도록 요청
4. 답변 + 검색된 context 반환 (평가에 필요)


In [12]:
RAG_PROMPT_TEMPLATE = """
You are a strict assistant.
Answer the question using ONLY the provided context.
If the context does not contain enough information, say: "I don't have enough information from the provided document."

Question:
{question}

Context:
{context}
""".strip()


In [13]:
def answer_with_rag(question: str, retriever):
    retrieved_docs = retriever.invoke(question)
    contexts = [d.page_content for d in retrieved_docs]
    joined_context = "\n\n---\n\n".join(contexts)

    prompt = RAG_PROMPT_TEMPLATE.format(question=question, context=joined_context)
    answer_text = call_ollama_runner(prompt)

    return {
        "question": question,
        "answer": answer_text,
        "contexts": contexts,
    }


In [14]:
test_question = "What is the purpose of the IGF Code and who is expected to follow it?"
example_result = answer_with_rag(test_question, retriever)

print("질문:", example_result["question"])
print("\n답변:\n", example_result["answer"])
print("\n검색된 context 수:", len(example_result["contexts"]))


질문: What is the purpose of the IGF Code and who is expected to follow it?

답변:
 The purpose of the IGF Code is to address all areas that need special consideration for the usage of low-flashpoint fuel. 

Contracting Governments to the Convention are expected to follow it.

검색된 context 수: 4


## Step 6 — Ground truth 데이터셋 (개요)

**ground truth** 행은 **`question`**과 RAG 시스템을 채점할 때 **정답 참조 답변** 역할을 하는 **`ground_truth`** 문자열을 짝지웁니다. 라벨이 약하거나 편향되면, 지표는 파이프라인보다 라벨 집합에 대해 더 많이 말해 줍니다.

**규모(프로덕션):** 필요한 양은 도메인, 리스크, 예산에 따라 다르며, 팀은 **수십~수백만** 항목을 라벨링할 수 있습니다. 라벨링이 일관되고 감사 가능하다면 **커버리지**(주제, 난이도, 엣지 케이스)가 단순 개수만큼 중요합니다.

**행별 필드 (Step 7–8에서 사용):**

- `question` — 평가 대상 RAG 시스템에 묻는 질문
- `ground_truth` — Ragas가 모델 답변과 비교하는 참조 답변 (**수동** 라벨은 Step 6a, 선택적 **LLM 합성** 테이블은 Step 6b에서 별도 채점)

이 단계는 라벨 **생성 방식**을 선택할 수 있게 나뉩니다:

- **Step 6a — 방법 1 (수동):** 사람이 작성한 라벨을 **`ground_truth_dataset_manual.csv`**로보냄. **Step 7은 해당 파일**(및 LLM draft CSV)을 로드하며, 메모리 내 Python 리스트는 사용하지 않음.
- **Step 6b — 방법 2 (PDF에서 LLM 합성):** 생성할 행 **개수**만 설정. LLM이 먼저 무작위 **PDF chunk**(`chunks`, Step 3)에서 **모든 질문**을 완성한 뒤, **별도 패스**에서 동일 chunk 텍스트와 **고정된 질문**으로 **답변**을 작성. **방법 2에는 Step 6a가 필수 아님** — Step 3으로 `chunks`만 있으면 됨.

#### 방법 1 vs 방법 2 — 장단점

| | **방법 1 — 수동 (6a)** | **방법 2 — PDF chunk 기반 LLM (6b)** |
|---|---------------------------|----------------------------------------|
| **장점** | 최고 충실도·감사 가능성; “모델이 자기 자신을 채점” 회피; 규제·고위험 도메인에 적합. | (question, answer) 쌍을 빠르게 부트스트랩; 질문이 샘플링한 **실제** chunk 텍스트에 근거; 실습·커버리지 탐색에 유리. |
| **단점** | 대규모에서 느리고 비용 큼; 일관성을 위해 가이드라인과 (이상적으로) 2차 검수자 필요. | PDF 대조 전까지 라벨은 **초안**; 환각·느슨한 의역 위험; 라벨러 모델 = RAG 모델이면 **순환 편향** (`OLLAMA_GT_LABEL_MODEL` 또는 별도 teacher로 완화). |
| **의존** | 원문 PDF(또는 루브릭) 읽기. | Step 3 chunking (`chunks`); 6b의 로컬 Ollama (또는 선택적 OpenAI 경로 6c). |

**선택 Step 6c**는 API 자격 증명이 있을 때 호스팅 `ChatOpenAI` 라벨러로 **6b와 동일한 2단계 레시피**를 반복 — Step 6a와 무관.


### Step 6a — 방법 1: 수동 ground truth 큐레이션

**목표:** 원문(여기서는 IGF Code PDF)을 읽고 질문과 짧은 사실 기반 답변을 직접 작성(또는 강사 제공 루브릭 사용)해 **신뢰할 수 있는** 평가 테이블을 구축합니다.

| 항목 | 내용 |
|-------|---------|
| **워크플로** | 구절 읽기 → `question` 작성 → 해당 구절에 엄격히 근거한 `ground_truth` 작성 → (권장) 페이지/절 메모 → 선택적 2차 검수자 |
| **장점** | 최고 라벨 충실도; “모델이 자기 자신을 채점” 회피; 고위험 도메인에 필수 |
| **한계** | 대규모에서 느리고 비용 큼; 여러 사람이 일관되게 라벨하려면 주석 가이드라인 필요 |

다음 셀의 리스트는 **시작 템플릿**일 뿐입니다. 본격 평가 시 PDF에서 근거 있는 항목으로 교체하세요.


In [15]:
from IPython.display import display

# --- Step 6a: manually curated ground truth (saved to CSV; Step 7 loads that file) ---
manual_ground_truth_data = [
    {
        "question": "What is the main objective of the IGF Code?",
        "ground_truth": "The IGF Code establishes standards and guidance to ensure integrity, governance, and responsible conduct in the relevant IGF framework.",
        "label_source": "manual",
    },
    {
        "question": "Who are the intended stakeholders or users of the IGF Code?",
        "ground_truth": "The Code is intended for organizations, professionals, and stakeholders expected to align their behavior and decisions with the IGF principles.",
        "label_source": "manual",
    },
    {
        "question": "What kinds of behavior or practices does the Code try to prevent?",
        "ground_truth": "It seeks to prevent misconduct, unethical behavior, conflicts of interest, and practices that undermine integrity and compliance.",
        "label_source": "manual",
    },
    {
        "question": "Does the Code include governance or compliance expectations? If yes, what is the intent?",
        "ground_truth": "Yes. It provides governance and compliance expectations to make responsibilities clear, improve accountability, and reduce policy violations.",
        "label_source": "manual",
    },
    {
        "question": "How does the Code describe accountability when rules are not followed?",
        "ground_truth": "The Code emphasizes accountability mechanisms so non-compliance can be identified, addressed, and corrected through defined processes.",
        "label_source": "manual",
    },
    {
        "question": "What is the expected benefit of following the IGF Code consistently?",
        "ground_truth": "Consistent adherence improves trust, transparency, and responsible decision-making across the organizations and participants covered by the Code.",
        "label_source": "manual",
    },
]

ground_truth_data = manual_ground_truth_data
ground_truth_df = pd.DataFrame(ground_truth_data)

# Step 7 reads ``outputs/ground_truth_dataset_manual.csv`` and ``outputs/ground_truth_dataset_llm_draft.csv`` (run the export cell above first).
# You can still use ``ground_truth_data`` / ``llm_draft_ground_truth_df`` in-session; CSV is the source of truth for evaluation reruns.

print("수동 ground truth 행 수:", len(ground_truth_df))
display(ground_truth_df)


수동 ground truth 행 수: 6


,question,ground_truth,label_source
0,What is the main objective of the IGF Code?,"The IGF Code establishes standards and guidance to ensure integrity, governance, and responsible conduct in the relevant IGF framework.",manual
1,Who are the intended stakeholders or users of the IGF Code?,"The Code is intended for organizations, professionals, and stakeholders expected to align their behavior and decisions with the IGF principles.",manual
2,What kinds of behavior or practices does the Code try to prevent?,"It seeks to prevent misconduct, unethical behavior, conflicts of interest, and practices that undermine integrity and compliance.",manual
3,"Does the Code include governance or compliance expectations? If yes, what is the intent?","Yes. It provides governance and compliance expectations to make responsibilities clear, improve accountability, and reduce policy violations.",manual
4,How does the Code describe accountability when rules are not followed?,"The Code emphasizes accountability mechanisms so non-compliance can be identified, addressed, and corrected through defined processes.",manual
5,What is the expected benefit of following the IGF Code consistently?,"Consistent adherence improves trust, transparency, and responsible decision-making across the organizations and participants covered by the Code.",manual


### Step 6b — 방법 2: PDF에서 LLM 합성 ground truth (로컬 Ollama)

**목표:** Step 3에서 이미 분할된 **실제 PDF 텍스트** `chunks`에서 **새** 평가 `(question, ground_truth)` 행을 생성합니다. **`N_SYNTH_GROUND_TRUTH`**(개수만)를 설정하면 노트북이 그만큼 chunk를 무작위 샘플링합니다.

**중요 — 명시적 2단계 LLM (질문 먼저, 답변 나중):**

1. **Phase 1 — 질문만**
   샘플링된 각 chunk에 대해 라벨러 LLM이 해당 chunk만으로 답할 수 있는 **질문 하나**를 작성합니다. **이 단계에서는 답변 프롬프트를 실행하지 않음** — 모든 질문이 생긴 뒤 실행이 “멈춤”.

2. **Phase 2 — 고정 질문에서 답변**
   **동일** chunk 텍스트와 **Phase 1 질문**(재샘플링 없음)으로 라벨러 LLM이 `ground_truth`를 작성합니다. 코드 출력에서 phase 사이에 질문 테이블을 검토할 수 있습니다.

이 설계는 흔한 평가 워크플로와 맞습니다: **질문 집합을 고정**한 뒤 참조 답변을 생성·수정해, 답변이 항상 고정된 question + passage 쌍에 조건화됩니다.

#### 라벨러 LLM vs RAG 생성 LLM (편향)

라벨링에는 RAG 답변과 **다른** 모델·제공자를 쓰는 것이 좋습니다 (`OLLAMA_GT_LABEL_MODEL`). 동일 가중치 재사용은 **수업용 단축**이며, 출판급 벤치마크에는 부적합합니다.

#### 환경 변수

| 변수 | 의미 |
|----------|---------|
| `N_SYNTH_GROUND_TRUTH` | 합성 행 수 (각 행 = chunk 1개, LLM 호출 2회: Q 후 A). |
| `SYNTH_GROUND_TRUTH_SEED` | 샘플링할 chunk RNG 시드. |
| `OLLAMA_GT_LABEL_MODEL` | 두 phase 모두용 선택 Ollama 태그. |
| `SKIP_STEP6B_SYNTH` 또는 `SKIP_LLM_DRAFT_GT` | `1`이면 Step 6b의 모든 Ollama 호출 건너뜀. |


In [16]:
# --- Step 6b: synthetic (question, ground_truth) from PDF chunks — two phases: ALL questions first, THEN answers ---
import inspect
import random

# === How many synthetic rows (each row = 1 random chunk, phase1 question + phase2 answer) ===
N_SYNTH_GROUND_TRUTH = 20
N_SYNTH_GROUND_TRUTH = int(
    os.getenv(
        "N_SYNTH_GROUND_TRUTH",
        os.getenv("N_STEP6B_SYNTH_PAIRS", str(N_SYNTH_GROUND_TRUTH)),
    )
)
SYNTH_GROUND_TRUTH_SEED = int(os.getenv("SYNTH_GROUND_TRUTH_SEED", "42"))
OLLAMA_GT_LABEL_MODEL = os.getenv("OLLAMA_GT_LABEL_MODEL", "").strip()

SYNTH_Q_PROMPT = """
You write evaluation questions for a regulatory / technical document.
Given PASSAGE text only, write ONE clear, specific question that can be answered using ONLY information in the passage.
Rules:
- Output ONLY the question text (no numbering, no quotes, no preamble).
- Do not mention "the passage" or "the document".

PASSAGE:
{passage}
""".strip()


SYNTH_A_PROMPT = """
You write reference answers for RAG evaluation (ground-truth style labels).
Given PASSAGE and a QUESTION, write ONE concise factual paragraph answering the question using ONLY the passage.
Rules:
- If the passage does not support an answer, reply with exactly this single line: INSUFFICIENT_CONTEXT
- Max ~120 words, plain text, no bullet list.

QUESTION:
{question}

PASSAGE:
{passage}
""".strip()



def _trim_passage(passage: str, limit: int = 12000) -> str:
    t = passage.strip()
    if len(t) > limit:
        return t[:limit] + "\n\n[...truncated for labeling prompt...]"
    return t


def _call_ollama_runner_for_gt_labeling(prompt: str, labeler_models: str | None) -> str:
    sig = inspect.signature(call_ollama_runner)
    if "models" in sig.parameters:
        return call_ollama_runner(prompt, models=labeler_models)
    if labeler_models:
        raise RuntimeError(
            "OLLAMA_GT_LABEL_MODEL is set but call_ollama_runner has no models= argument. "
            "Re-run the setup cell that defines call_ollama_runner, or unset OLLAMA_GT_LABEL_MODEL."
        )
    return call_ollama_runner(prompt)


SKIP_STEP6B_SYNTH = os.getenv("SKIP_STEP6B_SYNTH", os.getenv("SKIP_LLM_DRAFT_GT", "0")).lower() in (
    "1",
    "true",
    "yes",
)

_labeler_models = OLLAMA_GT_LABEL_MODEL or None
if _labeler_models is None:
    print("OLLAMA_GT_LABEL_MODEL이 비어 있음 — 합성기가 OLLAMA_MODELS 사용 (수업용 단축).")
else:
    print("전용 합성 모델 사용:", _labeler_models)

if "chunks" not in globals() or not chunks:
    raise RuntimeError("Run Step 3 (chunking) first so `chunks` exists — chunks come from the PDF.")

if SKIP_STEP6B_SYNTH:
    llm_draft_ground_truth_data = []
    llm_draft_ground_truth_df = pd.DataFrame(
        columns=[
            "question",
            "ground_truth",
            "source_page",
            "passage_chars",
            "label_source",
            "labeler_models",
        ]
    )
    print("SKIP_STEP6B_SYNTH / SKIP_LLM_DRAFT_GT 설정됨 — Step 6b 건너뜀.")
else:
    rng = random.Random(SYNTH_GROUND_TRUTH_SEED)
    _chunk_list = list(chunks)
    n_pick = min(N_SYNTH_GROUND_TRUTH, len(_chunk_list))
    picked = rng.sample(_chunk_list, n_pick)
    passages_raw = [c.page_content for c in picked]
    page_hints = [c.metadata.get("page") for c in picked]
    passages = [_trim_passage(p) for p in passages_raw]

    # ----- Phase 1: generate ALL questions from real chunk text; no answer LLM calls yet -----
    synth_questions: list[str] = []
    for passage_trim, page_hint in tqdm(
        list(zip(passages, page_hints)),
        desc="Step 6b phase 1 — questions only (PDF chunks)",
    ):
        q_prompt = SYNTH_Q_PROMPT.format(passage=passage_trim)
        question = _call_ollama_runner_for_gt_labeling(q_prompt, _labeler_models).strip()
        synth_questions.append(question)

    print("Phase 1 완료:", len(synth_questions), "개 질문,", n_pick, "개 PDF chunk에서 생성 (답변 전 LLM 일시 중지).")
    display(
        pd.DataFrame(
            {
                "question": synth_questions,
                "source_page": page_hints,
                "passage_chars": [len(p) for p in passages],
            }
        )
    )

    # ----- Phase 2: answers only, each conditioned on frozen question + same passage -----
    rows: list[dict] = []
    for question, passage_trim, page_hint in tqdm(
        list(zip(synth_questions, passages, page_hints)),
        desc="Step 6b phase 2 — answers from frozen questions + passages",
    ):
        a_prompt = SYNTH_A_PROMPT.format(question=question, passage=passage_trim)
        ground_truth = _call_ollama_runner_for_gt_labeling(a_prompt, _labeler_models).strip()
        rows.append(
            {
                "question": question,
                "ground_truth": ground_truth,
                "source_page": page_hint,
                "passage_chars": len(passage_trim),
                "label_source": "step6b_synthetic_local",
                "labeler_models": _labeler_models or OLLAMA_MODELS,
            }
        )

    llm_draft_ground_truth_data = rows
    llm_draft_ground_truth_df = pd.DataFrame(rows)

print("요청한 합성 행 수:", N_SYNTH_GROUND_TRUTH, "| 생성됨:", len(llm_draft_ground_truth_df))
display(llm_draft_ground_truth_df)

if "openai_gt_draft_df" not in globals():
    openai_gt_draft_df = pd.DataFrame(
        columns=["question", "ground_truth", "source_page", "passage_chars", "label_source", "labeler_model"]
    )


OLLAMA_GT_LABEL_MODEL이 비어 있음 — 합성기가 OLLAMA_MODELS 사용 (수업용 단축).


Step 6b phase 1 — questions only (PDF chunks):   0%|          | 0/20 [00:00<?, ?it/s]

Phase 1 완료: 20 개 질문, 20 개 PDF chunk에서 생성 (답변 전 LLM 일시 중지).


,question,source_page,passage_chars
0,What method shall be used to carry out buckling strength analyses of fuel tanks subject to external pressure and other loads causing compressive s...,101,965
1,What is the minimum value of fCN to be used in case the fuel tank arrangement is unsymmetrical about the centreline of the ship?,16,946
2,What is the page number where regulations for gas fuel supply to consumers in ESD-protected machinery spaces are located?,2,378
3,What is the minimum design pressure that must be applied during a pressure test of a type expansion joint?,116,961
4,What is the maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces?,43,994
5,"What is the minimum design temperature that a longitudinal bulkhead between liquefied gas fuel tanks must remain suitable for, in order to take cr...",39,933
6,What are the six types of total stress values that shall be determined for equivalent stress calculation?,35,990
7,"What potential dangers to the ship, persons on board, or environment should be avoided when designing the fuel containment system?",20,926
8,What percentage of butt-welded joints of pipes must be subjected to radiographic or ultrasonic inspection?,115,787
9,What is the minimum distance that the lowermost boundary of the fuel tank(s) must be located above?,14,504


Step 6b phase 2 — answers from frozen questions + passages:   0%|          | 0/20 [00:00<?, ?it/s]

요청한 합성 행 수: 20 | 생성됨: 20


,question,ground_truth,source_page,passage_chars,label_source,labeler_models
0,What method shall be used to carry out buckling strength analyses of fuel tanks subject to external pressure and other loads causing compressive s...,"According to MSC 95/22/Add.1, buckling strength analyses of fuel tanks subject to external pressure and other loads causing compressive stresses s...",101,965,step6b_synthetic_local,llama3.1:8b
1,What is the minimum value of fCN to be used in case the fuel tank arrangement is unsymmetrical about the centreline of the ship?,"In case the fuel tank arrangement is unsymmetrical about the centreline of the ship, the calculations of fCN should be performed on both starboard...",16,946,step6b_synthetic_local,llama3.1:8b
2,What is the page number where regulations for gas fuel supply to consumers in ESD-protected machinery spaces are located?,"The regulations for gas fuel supply to consumers in ESD-protected machinery spaces are located on page 79, as indicated by the ellipsis leading up...",2,378,step6b_synthetic_local,llama3.1:8b
3,What is the minimum design pressure that must be applied during a pressure test of a type expansion joint?,The minimum design pressure that must be applied during a pressure test of a type expansion joint is twice the design pressure at the extreme disp...,116,961,step6b_synthetic_local,llama3.1:8b
4,What is the maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces?,The maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces is less than 0.07 MPa.,43,994,step6b_synthetic_local,llama3.1:8b
5,"What is the minimum design temperature that a longitudinal bulkhead between liquefied gas fuel tanks must remain suitable for, in order to take cr...","For a longitudinal bulkhead between liquefied gas fuel tanks, credit may be taken for heating if the material remains suitable for a minimum desig...",39,933,step6b_synthetic_local,llama3.1:8b
6,What are the six types of total stress values that shall be determined for equivalent stress calculation?,"The six types of total stress values that shall be determined for equivalent stress calculation are: total normal stress in x-direction (σx), tota...",35,990,step6b_synthetic_local,llama3.1:8b
7,"What potential dangers to the ship, persons on board, or environment should be avoided when designing the fuel containment system?","When designing the fuel containment system, potential dangers to avoid include exposure of ship materials to temperatures below acceptable limits,...",20,926,step6b_synthetic_local,llama3.1:8b
8,What percentage of butt-welded joints of pipes must be subjected to radiographic or ultrasonic inspection?,At least 10% of butt-welded joints of pipes must be subjected to radiographic or ultrasonic inspection.,115,787,step6b_synthetic_local,llama3.1:8b
9,What is the minimum distance that the lowermost boundary of the fuel tank(s) must be located above?,"The minimum distance that the lowermost boundary of the fuel tank(s) must be located above is 2.0 m or B/15, whichever is less.",14,504,step6b_synthetic_local,llama3.1:8b


### 선택 — Step 6c: 동일 2단계 합성 ground truth (OpenAI API)

이 블록은 **선택**이며 **기본 꺼짐**. **Step 6b**를 반영합니다: **Phase 1**이 무작위 PDF `chunks`에서 **모든** 질문을 쓰고, **Phase 2**가 고정 질문과 동일 chunk 텍스트로 답변을 씁니다 — 로컬 runner 대신 `ChatOpenAI` 사용.

**Step 6b에 의존하지 않음:** 프롬프트 템플릿은 아래 코드 셀에 다시 정의됨 (6b 텍스트 복제). **`chunks`를 위해 Step 3**은 여전히 필요.

| 변수 | 용도 |
|----------|---------|
| `USE_OPENAI_GT_LABELER` | `1`이면 이 블록 실행. |
| `OPENAI_API_KEY` | OpenAI 호환 서비스 API 키. |
| `N_SYNTH_GROUND_TRUTH` | Step 6b와 동일 행 수 (코드/환경에서 덮어쓰기 가능). |
| `SYNTH_GROUND_TRUTH_SEED` | Step 6b와 동일 chunk 샘플링 시드 의미. |
| `OPENAI_GT_MODEL` | Chat 모델 id (기본 `gpt-4o-mini`). |
| `OPENAI_GT_BASE_URL` | Chat Completions base URL. |

> 둘 다 켜면 OpenAI vs 로컬 draft를 비교할 수 있으나, PDF 대조 후 사람 검수를 거쳐 gold로 승격하세요.


In [17]:
# OPTIONAL — Step 6c: two-phase synthetic Q/A via OpenAI-compatible API (standalone prompts).
import random

USE_OPENAI_GT_LABELER = os.getenv("USE_OPENAI_GT_LABELER", "0").lower() in ("1", "true", "yes")
OPENAI_GT_MODEL = os.getenv("OPENAI_GT_MODEL", "gpt-4o-mini")
OPENAI_GT_BASE_URL = os.getenv("OPENAI_GT_BASE_URL", os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1"))
OPENAI_GT_API_KEY = os.getenv("OPENAI_API_KEY", "")

N_SYNTH_GROUND_TRUTH = 20
N_SYNTH_GROUND_TRUTH = int(
    os.getenv(
        "N_SYNTH_GROUND_TRUTH",
        os.getenv("N_STEP6C_SYNTH_PAIRS", os.getenv("N_STEP6B_SYNTH_PAIRS", str(N_SYNTH_GROUND_TRUTH))),
    )
)
SYNTH_GROUND_TRUTH_SEED = int(
    os.getenv("SYNTH_GROUND_TRUTH_SEED", os.getenv("SYNTH6C_RANDOM_SEED", os.getenv("SYNTH6B_RANDOM_SEED", "42")))
)

SYNTH_Q_PROMPT = """
You write evaluation questions for a regulatory / technical document.
Given PASSAGE text only, write ONE clear, specific question that can be answered using ONLY information in the passage.
Rules:
- Output ONLY the question text (no numbering, no quotes, no preamble).
- Do not mention "the passage" or "the document".

PASSAGE:
{passage}
""".strip()


SYNTH_A_PROMPT = """
You write reference answers for RAG evaluation (ground-truth style labels).
Given PASSAGE and a QUESTION, write ONE concise factual paragraph answering the question using ONLY the passage.
Rules:
- If the passage does not support an answer, reply with exactly this single line: INSUFFICIENT_CONTEXT
- Max ~120 words, plain text, no bullet list.

QUESTION:
{question}

PASSAGE:
{passage}
""".strip()



def _trim_passage_c(passage: str, limit: int = 12000) -> str:
    t = passage.strip()
    if len(t) > limit:
        return t[:limit] + "\n\n[...truncated for labeling prompt...]"
    return t


if USE_OPENAI_GT_LABELER:
    if not OPENAI_GT_API_KEY:
        raise RuntimeError("USE_OPENAI_GT_LABELER is set but OPENAI_API_KEY is empty.")
    if "chunks" not in globals() or not chunks:
        raise RuntimeError("Run Step 3 so `chunks` exists before Step 6c.")

    llm_labeler = ChatOpenAI(
        model=OPENAI_GT_MODEL,
        api_key=OPENAI_GT_API_KEY,
        base_url=OPENAI_GT_BASE_URL,
        temperature=0,
    )

    rng = random.Random(SYNTH_GROUND_TRUTH_SEED)
    _chunk_list = list(chunks)
    n_pick = min(N_SYNTH_GROUND_TRUTH, len(_chunk_list))
    picked = rng.sample(_chunk_list, n_pick)
    passages_raw = [c.page_content for c in picked]
    page_hints = [c.metadata.get("page") for c in picked]
    passages = [_trim_passage_c(p) for p in passages_raw]

    synth_questions: list[str] = []
    for passage_trim, page_hint in tqdm(
        list(zip(passages, page_hints)),
        desc="Step 6c phase 1 — questions only (PDF chunks)",
    ):
        q_prompt = SYNTH_Q_PROMPT.format(passage=passage_trim)
        synth_questions.append(llm_labeler.invoke(q_prompt).content.strip())

    print("Phase 1 완료:", len(synth_questions), "개 질문 (OpenAI); Phase 2 — 답변 시작.")
    display(
        pd.DataFrame(
            {
                "question": synth_questions,
                "source_page": page_hints,
                "passage_chars": [len(p) for p in passages],
            }
        )
    )

    rows: list[dict] = []
    for question, passage_trim, page_hint in tqdm(
        list(zip(synth_questions, passages, page_hints)),
        desc="Step 6c phase 2 — answers from frozen questions + passages",
    ):
        a_prompt = SYNTH_A_PROMPT.format(question=question, passage=passage_trim)
        ground_truth = llm_labeler.invoke(a_prompt).content.strip()
        rows.append(
            {
                "question": question,
                "ground_truth": ground_truth,
                "source_page": page_hint,
                "passage_chars": len(passage_trim),
                "label_source": "step6c_synthetic_openai",
                "labeler_model": OPENAI_GT_MODEL,
            }
        )

    openai_gt_draft_df = pd.DataFrame(rows)
    print("요청한 합성 행 수:", N_SYNTH_GROUND_TRUTH, "| 생성됨:", len(openai_gt_draft_df))
    display(openai_gt_draft_df.head())
else:
    openai_gt_draft_df = pd.DataFrame(
        columns=[
            "question",
            "ground_truth",
            "source_page",
            "passage_chars",
            "label_source",
            "labeler_model",
        ]
    )
    print("Step 6c 비활성화 (실행하려면 USE_OPENAI_GT_LABELER=1 및 OPENAI_API_KEY 설정).")


Step 6c 비활성화 (실행하려면 USE_OPENAI_GT_LABELER=1 및 OPENAI_API_KEY 설정).


### Ground truth 품질 개선 (중요)

- **Step 6a (수동):** PDF의 정확한 구절을 읽고, 각 `ground_truth`를 근거에 밀착하도록 다시 쓰고, 내부 페이지/절 메모를 유지하며 가능하면 **2차 검수자**를 둡니다.
- **Step 6b (로컬 합성):** phase 1 후 생성 **질문**의 모호·주제 이탈을 훑고, phase 2 후 각 **답변**을 PDF chunk(필요 시 전체 context)와 대조합니다. gold로 쓰기 전 `INSUFFICIENT_CONTEXT`와 환각을 주의하세요.
- **Step 6c (OpenAI 합성, 선택):** 6b와 동일한 검수 원칙 + API 비용·데이터 처리 정책.

이 워크플로는 Ragas 점수와 파이프라인 변경에 대한 결론의 신뢰도를 실질적으로 높입니다.


In [18]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

manual_gt_path = output_dir / "ground_truth_dataset_manual.csv"
ground_truth_df.to_csv(manual_gt_path, index=False)

llm_draft_gt_path = output_dir / "ground_truth_dataset_llm_draft.csv"
llm_draft_ground_truth_df.to_csv(llm_draft_gt_path, index=False)

if not openai_gt_draft_df.empty:
    openai_gt_path = output_dir / "ground_truth_dataset_openai_draft.csv"
    openai_gt_draft_df.to_csv(openai_gt_path, index=False)
    print("저장됨:", openai_gt_path)

print("저장됨:", manual_gt_path)
print("저장됨:", llm_draft_gt_path)


저장됨: outputs/ground_truth_dataset_manual.csv
저장됨: outputs/ground_truth_dataset_llm_draft.csv


## Step 7 — 평가 집합에 대한 RAG 답변 생성

ground truth 라벨은 Step 6에서 쓴 **두 CSV 파일**에서 읽습니다 (`outputs/ground_truth_dataset_manual.csv`와 동일 폴더):

| 파일 | 역할 |
|------|------|
| `ground_truth_dataset_manual.csv` | 수동 질문 + 참조 답변 (Step 6a) |
| `ground_truth_dataset_llm_draft.csv` | LLM 합성 질문 + 참조 답변 (Step 6b) |

선택적 환경 변수 덮어쓰기 (이 셀 실행 전 경로가 존재해야 함):

- `GROUND_TRUTH_MANUAL_CSV` — 기본 `outputs/ground_truth_dataset_manual.csv`
- `GROUND_TRUTH_LLM_CSV` — 기본 `outputs/ground_truth_dataset_llm_draft.csv`

**각** 파일의 모든 행에 **동일** RAG 파이프라인을 실행합니다 (두 채점 트랙). Retriever 출력은 **질문**에만 의존하며, 지표용 참조 문자열은 CSV 행에서 옵니다.


In [19]:
# --- Step 7: load ground truth from CSV (manual + LLM), then run RAG for each track ---
import os


def _gt_rows_from_csv(path: Path, *, label: str) -> list[dict]:
    """Load rows with ``question`` and ``ground_truth`` only."""
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {label} CSV: {path}. Run the Step 6 export cell first, or set GROUND_TRUTH_* env override."
        )
    df = pd.read_csv(path)
    need = {"question", "ground_truth"}
    if not need.issubset(df.columns):
        raise ValueError(f"{path} must contain columns {sorted(need)}; got {list(df.columns)}")
    df = df[list(need)].dropna(subset=list(need))
    return df.to_dict("records")


def _rag_eval_rows(gt_rows: list[dict], retriever, tqdm_desc: str) -> list[dict]:
    out: list[dict] = []
    for row in tqdm(gt_rows, desc=tqdm_desc):
        rag_out = answer_with_rag(row["question"], retriever)
        out.append(
            {
                "question": row["question"],
                "answer": rag_out["answer"],
                "contexts": rag_out["contexts"],
                "ground_truth": row["ground_truth"],
            }
        )
    return out


_gt_root = Path(os.getenv("GROUND_TRUTH_OUTPUT_DIR", "outputs"))
manual_gt_path = Path(os.getenv("GROUND_TRUTH_MANUAL_CSV", _gt_root / "ground_truth_dataset_manual.csv"))
llm_gt_path = Path(os.getenv("GROUND_TRUTH_LLM_CSV", _gt_root / "ground_truth_dataset_llm_draft.csv"))

manual_gt_rows = _gt_rows_from_csv(manual_gt_path, label="manual")

if llm_gt_path.is_file():
    _llm_df = pd.read_csv(llm_gt_path)
    if {"question", "ground_truth"}.issubset(_llm_df.columns):
        _llm_df = _llm_df[["question", "ground_truth"]].dropna()
        llm_gt_rows = _llm_df.to_dict("records")
    else:
        llm_gt_rows = []
        print("경고: LLM CSV에 question/ground_truth 열 없음 — LLM 트랙 건너뜀.")
else:
    llm_gt_rows = []
    print(f"참고: {llm_gt_path}에서 LLM CSV를 찾을 수 없음 — LLM 채점 트랙 건너뜀 (선택 파일).")

eval_df_manual = pd.DataFrame(
    _rag_eval_rows(manual_gt_rows, retriever, tqdm_desc="Step 7 RAG (manual GT CSV)")
)
eval_df_llm = (
    pd.DataFrame(_rag_eval_rows(llm_gt_rows, retriever, tqdm_desc="Step 7 RAG (LLM GT CSV)"))
    if llm_gt_rows
    else pd.DataFrame(columns=["question", "answer", "contexts", "ground_truth"])
)

eval_df = eval_df_manual
eval_rows = eval_df_manual.to_dict("records")

print("수동 GT 행 수:", len(eval_df_manual), "출처", manual_gt_path)
print("LLM GT 행 수:", len(eval_df_llm), "출처", llm_gt_path if llm_gt_path.is_file() else "(해당 없음)")

display(eval_df_manual[["question", "answer", "ground_truth"]].head())
if len(eval_df_llm):
    display(eval_df_llm[["question", "answer", "ground_truth"]].head())


Step 7 RAG (manual GT CSV):   0%|          | 0/6 [00:00<?, ?it/s]

Step 7 RAG (LLM GT CSV):   0%|          | 0/20 [00:00<?, ?it/s]

수동 GT 행 수: 6 출처 outputs/ground_truth_dataset_manual.csv
LLM GT 행 수: 20 출처 outputs/ground_truth_dataset_llm_draft.csv


,question,answer,ground_truth
0,What is the main objective of the IGF Code?,The main objective of the IGF Code is to ensure safety on ships that use gases or other low-flashpoint fuels. The Code addresses all areas that ne...,"The IGF Code establishes standards and guidance to ensure integrity, governance, and responsible conduct in the relevant IGF framework."
1,Who are the intended stakeholders or users of the IGF Code?,The intended stakeholders or users of the IGF Code are:\n\n* Contracting Governments to the Convention\n* Members of the Organization which are no...,"The Code is intended for organizations, professionals, and stakeholders expected to align their behavior and decisions with the IGF principles."
2,What kinds of behavior or practices does the Code try to prevent?,I don't have enough information from the provided document to answer your question about what kinds of behavior or practices the Code tries to pre...,"It seeks to prevent misconduct, unethical behavior, conflicts of interest, and practices that undermine integrity and compliance."
3,"Does the Code include governance or compliance expectations? If yes, what is the intent?",I don't have enough information from the provided document to answer whether the Code includes governance or compliance expectations and what is t...,"Yes. It provides governance and compliance expectations to make responsibilities clear, improve accountability, and reduce policy violations."
4,How does the Code describe accountability when rules are not followed?,I don't have enough information from the provided document to answer your question about how the Code describes accountability when rules are not ...,"The Code emphasizes accountability mechanisms so non-compliance can be identified, addressed, and corrected through defined processes."


,question,answer,ground_truth
0,What method shall be used to carry out buckling strength analyses of fuel tanks subject to external pressure and other loads causing compressive s...,The method to carry out buckling strength analyses of fuel tanks subject to external pressure and other loads causing compressive stresses is in a...,"According to MSC 95/22/Add.1, buckling strength analyses of fuel tanks subject to external pressure and other loads causing compressive stresses s..."
1,What is the minimum value of fCN to be used in case the fuel tank arrangement is unsymmetrical about the centreline of the ship?,The minimum value of fCN to be used in case the fuel tank arrangement is unsymmetrical about the centreline of the ship is not explicitly stated i...,"In case the fuel tank arrangement is unsymmetrical about the centreline of the ship, the calculations of fCN should be performed on both starboard..."
2,What is the page number where regulations for gas fuel supply to consumers in ESD-protected machinery spaces are located?,I don't have enough information from the provided document to determine the page number where regulations for gas fuel supply to consumers in ESD-...,"The regulations for gas fuel supply to consumers in ESD-protected machinery spaces are located on page 79, as indicated by the ellipsis leading up..."
3,What is the minimum design pressure that must be applied during a pressure test of a type expansion joint?,I don't have enough information from the provided document to answer your question about the minimum design pressure for a type expansion joint du...,The minimum design pressure that must be applied during a pressure test of a type expansion joint is twice the design pressure at the extreme disp...
4,What is the maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces?,The maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces is less than 0.07 MPa.,The maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces is less than 0.07 MPa.


In [20]:
raw_eval_path_manual = output_dir / "rag_eval_raw_outputs_manual_gt.csv"
temp_m = eval_df_manual.copy()
temp_m["contexts"] = temp_m["contexts"].apply(lambda x: "\n\n---\n\n".join(x))
temp_m.to_csv(raw_eval_path_manual, index=False)
print("저장됨:", raw_eval_path_manual)

if len(eval_df_llm):
    raw_eval_path_llm = output_dir / "rag_eval_raw_outputs_llm_gt.csv"
    temp_l = eval_df_llm.copy()
    temp_l["contexts"] = temp_l["contexts"].apply(lambda x: "\n\n---\n\n".join(x))
    temp_l.to_csv(raw_eval_path_llm, index=False)
    print("저장됨:", raw_eval_path_llm)


저장됨: outputs/rag_eval_raw_outputs_manual_gt.csv
저장됨: outputs/rag_eval_raw_outputs_llm_gt.csv


## Step 8 — Ragas 지표로 평가

**Faithfulness**와 **Answer relevancy**를 **각** CSV ground truth 트랙(수동 먼저, LLM 파일에 행이 있으면 LLM)에 대해 계산합니다. 참조 라벨 출처가 바뀔 때 점수가 어떻게 변하는지 집계를 비교하세요.

### 사용 지표

- **Faithfulness** (`faithfulness`): RAG 답변이 **검색된 context에 의해 뒷받침되는지** 측정.
  - 이 지표의 질문: *"답변이 검색 증거에 근거하지 않은 주장을 추가했는가?"*
  - 높은 점수: 답변 문장 대부분이 context로 추적 가능.
  - 낮은 점수: 환각, 과도한 추론, 약한 retrieval 커버리지를 시사.

- **Answer Relevancy** (`answer_relevancy`): 답변이 **사용자 질문을 얼마나 잘 다루는지** 측정.
  - 이 지표의 질문: *"답변이 실제로 묻은 내용에 응답하는가?"*
  - 높은 점수: 질문 의도에 초점.
  - 낮은 점수: 주제 이탈, 모호함, 부분적 응답.

### 점수 해석

- 두 지표는 보통 **0~1** (높을수록 좋음).
- 평균만 믿지 말고 Step 9에서 저점 샘플을 검사해 구체적 실패 모드를 찾으세요.
- 점수 패턴별 빠른 읽기:
  - **Faithfulness 높음 + Relevancy 높음**: 건강 (근거·초점 모두 양호).
  - **Faithfulness 높음 + Relevancy 낮음**: 근거는 있으나 생성/프롬프트가 질문 의도를 놓침.
  - **Faithfulness 낮음 + Relevancy 높음**: 주제는 맞아 보이나 증거 부족 (환각 위험).
  - **Faithfulness 낮음 + Relevancy 낮음**: retrieval, 프롬프트 또는 둘 다의 광범위한 문제.

### 수동 GT vs LLM GT 비교 이유

- **수동 ground truth**는 최종 품질 결론에 더 신뢰할 수 있음.
- **LLM ground truth**는 평가 확장에 유리하나 라벨러 모델 편향을 물려받을 수 있음.
- 두 트랙이 크게 어긋나면 결정 전 고차이 샘플의 수동 검사를 우선하세요.


In [21]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy


/tmp/ipykernel_336672/4141761775.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
/tmp/ipykernel_336672/4141761775.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy


In [22]:
# --- Ragas baseline (retriever k from Step 4): manual CSV vs LLM CSV ---
ragas_ready_manual = eval_df_manual[["question", "answer", "contexts", "ground_truth"]].copy()
ragas_dataset_manual = Dataset.from_pandas(ragas_ready_manual, preserve_index=False)
print("수동 GT — Dataset:", ragas_dataset_manual)

result_manual = evaluate(
    dataset=ragas_dataset_manual,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=embeddings,
)
score_df_manual = result_manual.to_pandas()

if len(eval_df_llm) == 0:
    result_llm = None
    score_df_llm = pd.DataFrame(columns=["faithfulness", "answer_relevancy"])
    print("LLM Ragas 패스 건너뜀 (LLM CSV에서 로드된 행 없음).")
else:
    ragas_ready_llm = eval_df_llm[["question", "answer", "contexts", "ground_truth"]].copy()
    ragas_dataset_llm = Dataset.from_pandas(ragas_ready_llm, preserve_index=False)
    print("LLM GT — Dataset:", ragas_dataset_llm)

    result_llm = evaluate(
        dataset=ragas_dataset_llm,
        metrics=[faithfulness, answer_relevancy],
        llm=ragas_llm,
        embeddings=embeddings,
    )
    score_df_llm = result_llm.to_pandas()

result = result_manual
score_df = score_df_manual


수동 GT — Dataset: Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 6
})


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM GT — Dataset: Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 20
})


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

In [23]:
display(score_df_manual.head())
if len(score_df_llm):
    display(score_df_llm.head())


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy
0,What is the main objective of the IGF Code?,"[Convention, \n \nHAVING CONSIDERED, at its ninety-fifth session, the draft International Code of Safety for \nShips using Gases or other Low-flas...",The main objective of the IGF Code is to ensure safety on ships that use gases or other low-flashpoint fuels. The Code addresses all areas that ne...,"The IGF Code establishes standards and guidance to ensure integrity, governance, and responsible conduct in the relevant IGF framework.",1.0,1.0
1,Who are the intended stakeholders or users of the IGF Code?,"[Convention, \n \nHAVING CONSIDERED, at its ninety-fifth session, the draft International Code of Safety for \nShips using Gases or other Low-flas...",The intended stakeholders or users of the IGF Code are:\n\n* Contracting Governments to the Convention\n* Members of the Organization which are no...,"The Code is intended for organizations, professionals, and stakeholders expected to align their behavior and decisions with the IGF principles.",0.5,1.0
2,What kinds of behavior or practices does the Code try to prevent?,"[accordance with 18.4.3. \n \n.5 Where arrangements are provided for overriding the overflow control system, \nthey shall be such that inadvertent...",I don't have enough information from the provided document to answer your question about what kinds of behavior or practices the Code tries to pre...,"It seeks to prevent misconduct, unethical behavior, conflicts of interest, and practices that undermine integrity and compliance.",0.8,0.0
3,"Does the Code include governance or compliance expectations? If yes, what is the intent?","[5 REQUESTS the Secretary-General of the Organization to transmit certified copies of \nthe present resolution and the text of the IGF Code, conta...",I don't have enough information from the provided document to answer whether the Code includes governance or compliance expectations and what is t...,"Yes. It provides governance and compliance expectations to make responsibilities clear, improve accountability, and reduce policy violations.",0.0,0.0
4,How does the Code describe accountability when rules are not followed?,[6.9 Regulations for the maintaining of fuel storage condition .............................. 62 \n6.10 Regulations on atmospheric control within ...,I don't have enough information from the provided document to answer your question about how the Code describes accountability when rules are not ...,"The Code emphasizes accountability mechanisms so non-compliance can be identified, addressed, and corrected through defined processes.",0.5,0.0


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy
0,What method shall be used to carry out buckling strength analyses of fuel tanks subject to external pressure and other loads causing compressive s...,"[MSC 95/22/Add.1 \nAnnex 1, page 51 \n \n \nI:\MSC\95\MSC 95-22-ADD.1 (E).docx \n6.4.15.3.3.2 Buckling criteria shall be as follows: \n \nThe thic...",The method to carry out buckling strength analyses of fuel tanks subject to external pressure and other loads causing compressive stresses is in a...,"According to MSC 95/22/Add.1, buckling strength analyses of fuel tanks subject to external pressure and other loads causing compressive stresses s...",0.857143,0.873068
1,What is the minimum value of fCN to be used in case the fuel tank arrangement is unsymmetrical about the centreline of the ship?,"[whichever is less, as required by 5.3.3.1. \n \n.2 For cargo ships: \n \n.1 for Vc below or equal 1,000 m3, 0.8 m; \n \n.2 for 1,000 m3 < Vc < 5,...",The minimum value of fCN to be used in case the fuel tank arrangement is unsymmetrical about the centreline of the ship is not explicitly stated i...,"In case the fuel tank arrangement is unsymmetrical about the centreline of the ship, the calculations of fCN should be performed on both starboard...",0.666667,0.000000
2,What is the page number where regulations for gas fuel supply to consumers in ESD-protected machinery spaces are located?,[9.5 Regulations for fuel distribution outside of machinery space ........................ 77 \n9.6 Regulations for fuel supply to consumers in ga...,I don't have enough information from the provided document to determine the page number where regulations for gas fuel supply to consumers in ESD-...,"The regulations for gas fuel supply to consumers in ESD-protected machinery spaces are located on page 79, as indicated by the ellipsis leading up...",0.333333,0.000000
3,What is the minimum design pressure that must be applied during a pressure test of a type expansion joint?,"[.2 A pressure test shall be performed on a type expansion joint, complete with \nall the accessories such as flanges, stays and articulations, at...",I don't have enough information from the provided document to answer your question about the minimum design pressure for a type expansion joint du...,The minimum design pressure that must be applied during a pressure test of a type expansion joint is twice the design pressure at the extreme disp...,0.000000,0.000000
4,What is the maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces?,"[given in 6.4.15 for the various tank types, a vapour pressure 𝑃ℎ higher than \nP0 may be accepted for site specific conditions (harbour or other...",The maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces is less than 0.07 MPa.,The maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces is less than 0.07 MPa.,0.333333,0.911932


In [24]:
summary = {
    "faithfulness_mean_manual_gt": float(score_df_manual["faithfulness"].mean()),
    "answer_relevancy_mean_manual_gt": float(score_df_manual["answer_relevancy"].mean()),
}
if len(score_df_llm):
    summary["faithfulness_mean_llm_gt"] = float(score_df_llm["faithfulness"].mean())
    summary["answer_relevancy_mean_llm_gt"] = float(score_df_llm["answer_relevancy"].mean())
else:
    summary["faithfulness_mean_llm_gt"] = None
    summary["answer_relevancy_mean_llm_gt"] = None

summary


{'faithfulness_mean_manual_gt': 0.5222222222222223,
 'answer_relevancy_mean_manual_gt': 0.3333333333333333,
 'faithfulness_mean_llm_gt': 0.5363408521303258,
 'answer_relevancy_mean_llm_gt': 0.46101558085109506}

In [25]:
pd.DataFrame([summary]).T


,0
faithfulness_mean_manual_gt,0.522222
answer_relevancy_mean_manual_gt,0.333333
faithfulness_mean_llm_gt,0.536341
answer_relevancy_mean_llm_gt,0.461016


## Step 9 — 저점 샘플 검사 (환각 분석)

집계 평균도 유용하지만 **행 단위 검사**에서 구체적 실패 모드를 찾습니다.

### 이 단계가 하는 일

1. 다음을 병합해 행별 분석 테이블 구축:
   - Step 7 RAG 출력 (`eval_df_manual`, `eval_df_llm`)
   - Step 8 Ragas 점수 (`score_df_manual`, `score_df_llm`)
2. 각 테이블을 낮은 `faithfulness`, 낮은 `answer_relevancy` 순으로 정렬.
3. 한 행을 상세 검사 (질문, 답변, ground truth, 점수, 검색 context).

### 효과적으로 쓰는 방법

- **매우 낮은 faithfulness** 행부터 근거 없는 주장을 찾습니다.
- 이어서 **낮은 relevancy** 행으로 주제 이탈 답변을 잡습니다.
- 수동 GT와 LLM GT 트랙에서 **동일 패턴**을 비교합니다.
- 두 트랙 모두 같은 질문을 약하게 표시하면 파이프라인 부채로 우선 처리하세요.

### 흔한 진단 패턴

- **faithfulness 낮음 + relevancy 높음**: 맞는 것 같지만 검색 증거에 근거하지 않음.
- **faithfulness 높음 + relevancy 낮음**: 근거는 있으나 질문 의도를 다루지 않음.
- **둘 다 낮음**: retrieval 품질, 프롬프트 제약 또는 둘 다 개선 필요.


In [26]:
def _build_analysis_df(eval_df: pd.DataFrame, score_df: pd.DataFrame, track_name: str) -> pd.DataFrame:
    if eval_df.empty or score_df.empty:
        return pd.DataFrame()

    out = pd.concat(
        [
            eval_df.reset_index(drop=True),
            score_df[["faithfulness", "answer_relevancy"]].reset_index(drop=True),
        ],
        axis=1,
    )
    out["gt_track"] = track_name
    out = out.sort_values(by=["faithfulness", "answer_relevancy"], ascending=[True, True]).reset_index(drop=True)
    return out


analysis_df_manual = _build_analysis_df(eval_df_manual, score_df_manual, "manual_csv")
analysis_df_llm = _build_analysis_df(eval_df_llm, score_df_llm, "llm_csv")

print("최저 점수 샘플 — 수동 GT (CSV):")
if analysis_df_manual.empty:
    print("수동 행 없음.")
else:
    display(
        analysis_df_manual[
            ["question", "answer", "ground_truth", "faithfulness", "answer_relevancy", "gt_track"]
        ].head(5)
    )

print("최저 점수 샘플 — LLM GT (CSV):")
if analysis_df_llm.empty:
    print("LLM 행 없음 (LLM CSV 없음 또는 비어 있음).")
else:
    display(
        analysis_df_llm[
            ["question", "answer", "ground_truth", "faithfulness", "answer_relevancy", "gt_track"]
        ].head(5)
    )

# Combined view for quick filtering/comparison across tracks.
analysis_df_all = pd.concat([analysis_df_manual, analysis_df_llm], ignore_index=True)

# Backward-compatible alias used by older cells.
analysis_df = analysis_df_manual if not analysis_df_manual.empty else analysis_df_llm


최저 점수 샘플 — 수동 GT (CSV):


,question,answer,ground_truth,faithfulness,answer_relevancy,gt_track
0,"Does the Code include governance or compliance expectations? If yes, what is the intent?",I don't have enough information from the provided document to answer whether the Code includes governance or compliance expectations and what is t...,"Yes. It provides governance and compliance expectations to make responsibilities clear, improve accountability, and reduce policy violations.",0.000000,0.0,manual_csv
1,What is the expected benefit of following the IGF Code consistently?,I don't have enough information from the provided document to answer your question. The context only mentions the adoption and implementation of t...,"Consistent adherence improves trust, transparency, and responsible decision-making across the organizations and participants covered by the Code.",0.333333,0.0,manual_csv
2,How does the Code describe accountability when rules are not followed?,I don't have enough information from the provided document to answer your question about how the Code describes accountability when rules are not ...,"The Code emphasizes accountability mechanisms so non-compliance can be identified, addressed, and corrected through defined processes.",0.500000,0.0,manual_csv
3,Who are the intended stakeholders or users of the IGF Code?,The intended stakeholders or users of the IGF Code are:\n\n* Contracting Governments to the Convention\n* Members of the Organization which are no...,"The Code is intended for organizations, professionals, and stakeholders expected to align their behavior and decisions with the IGF principles.",0.500000,1.0,manual_csv
4,What kinds of behavior or practices does the Code try to prevent?,I don't have enough information from the provided document to answer your question about what kinds of behavior or practices the Code tries to pre...,"It seeks to prevent misconduct, unethical behavior, conflicts of interest, and practices that undermine integrity and compliance.",0.800000,0.0,manual_csv


최저 점수 샘플 — LLM GT (CSV):


,question,answer,ground_truth,faithfulness,answer_relevancy,gt_track
0,What is the minimum design pressure that must be applied during a pressure test of a type expansion joint?,I don't have enough information from the provided document to answer your question about the minimum design pressure for a type expansion joint du...,The minimum design pressure that must be applied during a pressure test of a type expansion joint is twice the design pressure at the extreme disp...,0.000000,0.0,llm_csv
1,What are the six types of total stress values that shall be determined for equivalent stress calculation?,I don't have enough information from the provided document to answer your question about the six types of total stress values that shall be determ...,"The six types of total stress values that shall be determined for equivalent stress calculation are: total normal stress in x-direction (σx), tota...",0.000000,0.0,llm_csv
2,What types of instrumentation and electrical apparatus should be installed in areas described in point 23?,I don't have enough information from the provided document to answer your question about what types of instrumentation and electrical apparatus sh...,Instrumentation and electrical apparatus installed in the areas described in point 23 should be of a type suitable for zone 1. These areas include...,0.000000,0.0,llm_csv
3,What is the goal of ventilation as stated in section 13.1?,I don't have enough information from the provided document to answer your question about what is stated in section 13.1 regarding the goal of vent...,"The goal of ventilation, as stated in section 13.1, is to provide a safe and healthy working environment by removing or diluting hazardous substan...",0.000000,0.0,llm_csv
4,What is the page number where regulations for gas fuel supply to consumers in ESD-protected machinery spaces are located?,I don't have enough information from the provided document to determine the page number where regulations for gas fuel supply to consumers in ESD-...,"The regulations for gas fuel supply to consumers in ESD-protected machinery spaces are located on page 79, as indicated by the ellipsis leading up...",0.333333,0.0,llm_csv


In [27]:
track = "manual"  # choose: "manual", "llm", or "all"
row_id = 0

track_to_df = {
    "manual": analysis_df_manual,
    "llm": analysis_df_llm,
    "all": analysis_df_all,
}

if track not in track_to_df:
    raise ValueError(f"Invalid track={track!r}. Use one of {list(track_to_df)}")

picked_df = track_to_df[track]
if picked_df.empty:
    raise RuntimeError(f"No rows available for track={track!r}.")
if not (0 <= row_id < len(picked_df)):
    raise IndexError(f"row_id={row_id} out of range for track={track!r}; valid: 0..{len(picked_df)-1}")

row = picked_df.iloc[row_id]

print("트랙:", row.get("gt_track", track))
print("행:", row_id, "/", len(picked_df) - 1)
print("질문:\n", row["question"])
print("\n모델 답변:\n", row["answer"])
print("\nGround Truth:\n", row["ground_truth"])
print("\nFaithfulness:", row["faithfulness"])
print("Answer Relevancy:", row["answer_relevancy"])

print("\n첫 번째 검색 context 발췌:\n")
print(row["contexts"][0][:1500] if row["contexts"] else "context 없음")


트랙: manual_csv
행: 0 / 5
질문:
 Does the Code include governance or compliance expectations? If yes, what is the intent?

모델 답변:
 I don't have enough information from the provided document to answer whether the Code includes governance or compliance expectations and what is the intent.

Ground Truth:
 Yes. It provides governance and compliance expectations to make responsibilities clear, improve accountability, and reduce policy violations.

Faithfulness: 0.0
Answer Relevancy: 0.0

첫 번째 검색 context 발췌:

5 REQUESTS the Secretary-General of the Organization to transmit certified copies of 
the present resolution and the text of the IGF Code, contained in the annex, to all Contracting 
Governments to the Convention; 
 
6 REQUESTS ALSO the Secretary-General of the Org anization to transmit copies of 
the present resolution and the text of the IGF Code contained in the annex to all Members of 
the Organization which are not Contracting Governments to the SOLAS Convention.


## Step 10 — 파이프라인 개선 및 재평가

환각을 줄이기 위한 일반적 개선:

- retrieval 깊이(`k`) 증가
- chunking 개선 (`chunk_size`, `overlap`)
- 프롬프트에서 답변 제약 강화
- 증거 부족 시 더 엄격한 거절 동작

이 실습에서는 먼저 간단한 변경 하나를 시험합니다: `k`를 4에서 6으로 증가.


In [28]:
retriever_k6 = vector_db.as_retriever(search_kwargs={"k": 6})
print("새 Retriever top-k:", retriever_k6.search_kwargs.get("k"))


새 Retriever top-k: 6


In [29]:
eval_df_k6_manual = pd.DataFrame(
    _rag_eval_rows(manual_gt_rows, retriever_k6, tqdm_desc="Step 10 RAG k=6 (manual GT CSV)")
)
eval_df_k6_llm = (
    pd.DataFrame(_rag_eval_rows(llm_gt_rows, retriever_k6, tqdm_desc="Step 10 RAG k=6 (LLM GT CSV)"))
    if llm_gt_rows
    else pd.DataFrame(columns=["question", "answer", "contexts", "ground_truth"])
)

eval_df_k6 = eval_df_k6_manual
ragas_dataset_k6_manual = Dataset.from_pandas(
    eval_df_k6_manual[["question", "answer", "contexts", "ground_truth"]].copy(),
    preserve_index=False,
)
result_k6_manual = evaluate(
    dataset=ragas_dataset_k6_manual,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=embeddings,
)
score_df_k6_manual = result_k6_manual.to_pandas()

if llm_gt_rows:
    ragas_dataset_k6_llm = Dataset.from_pandas(
        eval_df_k6_llm[["question", "answer", "contexts", "ground_truth"]].copy(),
        preserve_index=False,
    )
    result_k6_llm = evaluate(
        dataset=ragas_dataset_k6_llm,
        metrics=[faithfulness, answer_relevancy],
        llm=ragas_llm,
        embeddings=embeddings,
    )
    score_df_k6_llm = result_k6_llm.to_pandas()
else:
    score_df_k6_llm = pd.DataFrame(columns=["faithfulness", "answer_relevancy"])

ragas_dataset_k6 = ragas_dataset_k6_manual
result_k6 = result_k6_manual
score_df_k6 = score_df_k6_manual


Step 10 RAG k=6 (manual GT CSV):   0%|          | 0/6 [00:00<?, ?it/s]

Step 10 RAG k=6 (LLM GT CSV):   0%|          | 0/20 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

In [30]:
_has_llm = len(score_df_llm) > 0

{
    "faithfulness_mean_k4_manual_gt": float(score_df_manual["faithfulness"].mean()),
    "answer_relevancy_mean_k4_manual_gt": float(score_df_manual["answer_relevancy"].mean()),
    "faithfulness_mean_k4_llm_gt": float(score_df_llm["faithfulness"].mean()) if _has_llm else None,
    "answer_relevancy_mean_k4_llm_gt": float(score_df_llm["answer_relevancy"].mean()) if _has_llm else None,
    "faithfulness_mean_k6_manual_gt": float(score_df_k6_manual["faithfulness"].mean()),
    "answer_relevancy_mean_k6_manual_gt": float(score_df_k6_manual["answer_relevancy"].mean()),
    "faithfulness_mean_k6_llm_gt": float(score_df_k6_llm["faithfulness"].mean()) if _has_llm else None,
    "answer_relevancy_mean_k6_llm_gt": float(score_df_k6_llm["answer_relevancy"].mean()) if _has_llm else None,
}


{'faithfulness_mean_k4_manual_gt': 0.5222222222222223,
 'answer_relevancy_mean_k4_manual_gt': 0.3333333333333333,
 'faithfulness_mean_k4_llm_gt': 0.5363408521303258,
 'answer_relevancy_mean_k4_llm_gt': 0.46101558085109506,
 'faithfulness_mean_k6_manual_gt': 0.5333333333333333,
 'answer_relevancy_mean_k6_manual_gt': 0.5888466473094519,
 'faithfulness_mean_k6_llm_gt': 0.5578947368421052,
 'answer_relevancy_mean_k6_llm_gt': 0.48913401581993465}

In [31]:
_has_llm = len(score_df_llm) > 0

comparison_df = pd.DataFrame(
    {
        "metric": ["faithfulness", "answer_relevancy"],
        "manual_gt_k4": [
            float(score_df_manual["faithfulness"].mean()),
            float(score_df_manual["answer_relevancy"].mean()),
        ],
        "manual_gt_k6": [
            float(score_df_k6_manual["faithfulness"].mean()),
            float(score_df_k6_manual["answer_relevancy"].mean()),
        ],
    }
)
comparison_df["delta_k6_vs_k4_manual"] = comparison_df["manual_gt_k6"] - comparison_df["manual_gt_k4"]

if _has_llm:
    comparison_df["llm_gt_k4"] = [
        float(score_df_llm["faithfulness"].mean()),
        float(score_df_llm["answer_relevancy"].mean()),
    ]
    comparison_df["llm_gt_k6"] = [
        float(score_df_k6_llm["faithfulness"].mean()),
        float(score_df_k6_llm["answer_relevancy"].mean()),
    ]
    comparison_df["delta_k6_vs_k4_llm"] = comparison_df["llm_gt_k6"] - comparison_df["llm_gt_k4"]
else:
    nan2 = [float("nan"), float("nan")]
    comparison_df["llm_gt_k4"] = nan2
    comparison_df["llm_gt_k6"] = nan2
    comparison_df["delta_k6_vs_k4_llm"] = nan2

comparison_df


,metric,manual_gt_k4,manual_gt_k6,delta_k6_vs_k4_manual,llm_gt_k4,llm_gt_k6,delta_k6_vs_k4_llm
0,faithfulness,0.522222,0.533333,0.011111,0.536341,0.557895,0.021554
1,answer_relevancy,0.333333,0.588847,0.255513,0.461016,0.489134,0.028118


## Step 11 — 평가 산출물 저장

여기서 쓰는 파일에는 Step 7의 트랙별 raw 출력과 **수동** vs **LLM** ground truth 출처로 구분된 분석 CSV(둘 다 Step 6보내기에서 로드)가 포함됩니다.


In [32]:
analysis_path_manual = output_dir / "rag_eval_analysis_manual_gt.csv"
analysis_df_manual.to_csv(analysis_path_manual, index=False)

if len(analysis_df_llm):
    analysis_path_llm = output_dir / "rag_eval_analysis_llm_gt.csv"
    analysis_df_llm.to_csv(analysis_path_llm, index=False)
else:
    analysis_path_llm = None

comparison_path = output_dir / "rag_eval_metric_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print("저장됨:")
print("-", analysis_path_manual)
if analysis_path_llm:
    print("-", analysis_path_llm)
print("-", comparison_path)


저장됨:
- outputs/rag_eval_analysis_manual_gt.csv
- outputs/rag_eval_analysis_llm_gt.csv
- outputs/rag_eval_metric_comparison.csv


## Step 12 — 학습자용 실습 체크리스트

수업/랩에서 이 체크리스트를 사용하세요:

1. 데이터 로드와 chunk 품질 검증.
2. 기준선 RAG 구축 및 점수 기록.
3. 더 높은 품질의 ground truth 샘플 큐레이션.
4. 낮은 faithfulness 사례를 먼저 검사.
5. 파이프라인 변경은 한 번에 하나씩.
6. 재평가 후 변화량(delta) 비교.
7. 재현성을 위해 실험 로그 유지.

이 루프를 반복하면 증거 품질과 retrieval 품질이 환각에 미치는 영향을 분명히 볼 수 있습니다.

---

### 권장 확장 실습

- 여러 chunking 전략 비교.
- 여러 프롬프트 템플릿 비교.
- embedding 모델 비교.
- 지표 추가 (예: context precision/recall).
- 실험 간 리더보드 테이블 구축.
